# Contact Document Generation from AFDB-1.6M

Read 100 CIFs from the afdb-1.6M dataset, compute atom/atom contacts (4.0 Å cutoff, heavy atoms),
and generate training documents using the deterministic-positives-only approach.

In [ ]:
import sys
sys.path.insert(0, "/home/ubuntu/contactdoc")

import pyarrow.parquet as pq
import pandas as pd
from pathlib import Path

from contactdoc.cif_parse import parse_cif, extract_residues
from contactdoc.contacts import compute_contacts, filter_contacts_by_plddt, sort_and_truncate
from contactdoc.serialize import serialize_document

## Load 100 entries from afdb-1.6M

In [ ]:
DATA_DIR = Path("/home/ubuntu/tim-us-east-1/datasets/afdb-1.6M")

# Read the first shard (2000 rows), take 100
shard_path = sorted(DATA_DIR.glob("shard_*.parquet"))[0]
print(f"Reading from: {shard_path}")

df = pq.read_table(shard_path).to_pandas().head(100)
print(f"Loaded {len(df)} entries")
df[["entry_id", "global_plddt", "seq_len", "struct_cluster_id", "split"]].head(10)

## Compute contacts and generate documents

Pipeline per entry:
1. Parse mmCIF string with Gemmi
2. Extract residues (1-based indexing, non-canonical → UNK)
3. Compute contacts (4.0 Å cutoff, heavy atoms only, no adjacent residues)
4. Filter contacts by per-residue pLDDT ≥ 70
5. Sort by sequence separation (longest-range first), truncate to 2048
6. Serialize to text document

In [ ]:
CUTOFF = 4.0
RESIDUE_PLDDT_MIN = 70.0
MAX_CONTACTS = 2048
TASK_TOKEN = "deterministic-positives-only"

documents = []
errors = []

for idx, row in df.iterrows():
    entry_id = row["entry_id"]
    cif_content = row["cif_content"]

    # 1. Parse
    try:
        structure = parse_cif(cif_content)
    except Exception as e:
        errors.append((entry_id, f"parse_error: {e}"))
        continue

    # 2. Extract residues
    result = extract_residues(structure)
    if isinstance(result, str):
        errors.append((entry_id, result))
        continue

    # 3. Compute contacts
    contacts = compute_contacts(result, CUTOFF)
    contacts_pre_filter = len(contacts)

    # 4. Filter by pLDDT
    contacts = filter_contacts_by_plddt(contacts, result, RESIDUE_PLDDT_MIN)
    if not contacts:
        errors.append((entry_id, "no_contacts_after_filter"))
        continue

    # 5. Sort and truncate
    contacts = sort_and_truncate(contacts, MAX_CONTACTS)

    # 6. Serialize
    doc_text = serialize_document(result.residues, contacts, task_token=TASK_TOKEN)

    documents.append({
        "entry_id": entry_id,
        "seq_len": len(result.residues),
        "contacts_pre_filter": contacts_pre_filter,
        "contacts_emitted": len(contacts),
        "doc_text": doc_text,
    })

print(f"Generated {len(documents)} documents, {len(errors)} errors")
if errors:
    print(f"\nFirst few errors:")
    for eid, reason in errors[:5]:
        print(f"  {eid}: {reason}")

## Inspect a sample document

In [ ]:
if documents:
    sample = documents[0]
    print(f"Entry: {sample['entry_id']}")
    print(f"Seq length: {sample['seq_len']}")
    print(f"Contacts (pre-filter): {sample['contacts_pre_filter']}")
    print(f"Contacts (emitted): {sample['contacts_emitted']}")
    print(f"\nDocument text (first 1000 chars):")
    print(sample["doc_text"][:1000])

## Summary statistics

In [ ]:
doc_df = pd.DataFrame(documents)
print("Document statistics:")
print(doc_df[["seq_len", "contacts_pre_filter", "contacts_emitted"]].describe().round(1))

print(f"\nDocument text lengths (chars):")
doc_df["doc_chars"] = doc_df["doc_text"].str.len()
print(doc_df["doc_chars"].describe().round(0))

## Tokenize documents

In [ ]:
from contactdoc.tokenizer import encode

token_lengths = []
for doc in documents:
    tokens = encode(doc["doc_text"])
    token_lengths.append(len(tokens))

token_df = pd.Series(token_lengths, name="token_count")
print("Token count statistics:")
print(token_df.describe().round(1))